# Optimisation des hyperparamètres - XGBoost (meilleure combinaison identifiée)

Module 1, point 9 du sujet (Khady KAMA) : optimisation via recherche
systématique (GridSearch ou Optuna). On utilise Optuna ici, plus efficace
qu'un GridSearch exhaustif sur ~1M lignes.

Modèle retenu : XGBoost + stratégie coût pondéré (meilleure combinaison
du tableau comparatif, AUC-PR = 0.024).

ATTENTION - contexte important (cf. note_methodologique_absence_signal.docx) :
Le diagnostic réalisé en amont a démontré que isFraud est statistiquement
indépendant des features disponibles sur ce dataset. L'optimisation des
hyperparamètres ne peut donc PAS créer de signal prédictif inexistant :
elle sert ici à (a) respecter la contrainte méthodologique du sujet, et
(b) vérifier, par l'absence de gain significatif malgré une recherche
poussée, une preuve supplémentaire de l'absence de signal (preuve n°7).
Un gain massif inattendu à cette étape remettrait en question le diagnostic
précédent et devrait être investigué avant d'aller plus loin.

Installation si besoin : pip install optuna

Auteur : Rasmané

In [ ]:
import pandas as pd
import numpy as np
import time
import warnings
import joblib

import optuna
from optuna.samplers import TPESampler
import xgboost as xgb
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
N_ESSAIS = 40  # nombre d'essais Optuna (compromis temps/exhaustivité sur 8 Go RAM)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

## 0. CHARGEMENT DES DONNÉES

In [ ]:
DOSSIER = r"C:\Users\hp\Documents\Fraude_detection\data"

X_train = pd.read_csv(f"{DOSSIER}/X_train.csv")
X_val = pd.read_csv(f"{DOSSIER}/X_val.csv")
y_train = pd.read_csv(f"{DOSSIER}/y_train.csv").squeeze()
y_val = pd.read_csv(f"{DOSSIER}/y_val.csv").squeeze()

ratio_desequilibre = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Ratio de déséquilibre (train) : {ratio_desequilibre:.1f} : 1")
print(f"X_train : {X_train.shape} | X_val : {X_val.shape}")

# Score de référence (résultat déjà obtenu sans optimisation, à battre)
AUC_PR_REFERENCE = 0.024
print(f"\nAUC-PR de référence (hyperparamètres par défaut) : {AUC_PR_REFERENCE}")

## 1. FONCTION OBJECTIF OPTUNA

In [ ]:
def objectif(trial):
    parametres = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "scale_pos_weight": ratio_desequilibre,  # fixé (stratégie coût pondéré)
        "random_state": SEED,
        "eval_metric": "aucpr",
        "n_jobs": -1,
        "verbosity": 0,
    }

    modele = xgb.XGBClassifier(**parametres)
    modele.fit(X_train, y_train)
    y_proba = modele.predict_proba(X_val)[:, 1]
    auc_pr = average_precision_score(y_val, y_proba)

    return auc_pr

## 2. LANCEMENT DE L'ÉTUDE OPTUNA

In [ ]:
print(f"\n{'='*70}")
print(f"LANCEMENT DE L'OPTIMISATION ({N_ESSAIS} essais)")
print(f"{'='*70}")

etude = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SEED),
    study_name="xgboost_fraude_optimisation"
)

debut = time.time()
etude.optimize(objectif, n_trials=N_ESSAIS, show_progress_bar=True)
duree_totale = time.time() - debut

print(f"\nOptimisation terminée en {duree_totale/60:.1f} minutes "
      f"({duree_totale/N_ESSAIS:.1f}s par essai en moyenne)")

## 3. RÉSULTATS

In [ ]:
print(f"\n{'='*70}")
print("RÉSULTATS DE L'OPTIMISATION")
print(f"{'='*70}")
print(f"Meilleur AUC-PR obtenu : {etude.best_value:.4f}")
print(f"AUC-PR de référence (sans optimisation) : {AUC_PR_REFERENCE}")

gain = etude.best_value - AUC_PR_REFERENCE
gain_relatif = (gain / AUC_PR_REFERENCE) * 100 if AUC_PR_REFERENCE > 0 else float("inf")
print(f"Gain absolu : {gain:+.4f}")
print(f"Gain relatif : {gain_relatif:+.1f}%")

print(f"\nMeilleurs hyperparamètres trouvés :")
for cle, valeur in etude.best_params.items():
    print(f"  {cle} : {valeur}")

print(f"\nSeuil exigé par le sujet : AUC-PR >= 0.90")
if etude.best_value >= 0.90:
    print("SEUIL ATTEINT (résultat inattendu au vu du diagnostic préalable")
    print("-> à re-vérifier immédiatement, possible fuite de données).")
else:
    ecart_au_seuil = 0.90 - etude.best_value
    print(f"Seuil NON atteint. Écart au seuil : {ecart_au_seuil:.4f}")
    print("Ce résultat est cohérent avec le diagnostic établi précédemment :")
    print("l'optimisation des hyperparamètres, même poussée, ne peut pas créer")
    print("de signal prédictif absent des données. Ce constat constitue une")
    print("preuve supplémentaire (preuve n°7) à ajouter à la note méthodologique.")

## 4. HISTORIQUE DES ESSAIS (pour analyse / graphique)

In [ ]:
historique = etude.trials_dataframe()
historique_utile = historique[["number", "value", "duration"] +
                               [c for c in historique.columns if c.startswith("params_")]]
historique_utile = historique_utile.sort_values("value", ascending=False)

print(f"\n{'='*70}")
print("TOP 5 DES ESSAIS")
print(f"{'='*70}")
print(historique_utile.head(5).to_string(index=False))

print(f"\nDispersion des scores sur les {N_ESSAIS} essais :")
print(historique_utile["value"].describe())
print("\n-> Une faible dispersion des scores (écart-type proche de 0) confirme")
print("   que les hyperparamètres n'ont quasiment aucun effet sur la")
print("   performance, ce qui est attendu en l'absence de signal exploitable.")

## 5. ENTRAÎNEMENT DU MODÈLE FINAL AVEC LES MEILLEURS HYPERPARAMÈTRES

In [ ]:
print(f"\n{'='*70}")
print("ENTRAÎNEMENT DU MODÈLE FINAL")
print(f"{'='*70}")

meilleurs_parametres = etude.best_params.copy()
meilleurs_parametres["scale_pos_weight"] = ratio_desequilibre
meilleurs_parametres["random_state"] = SEED
meilleurs_parametres["eval_metric"] = "aucpr"
meilleurs_parametres["n_jobs"] = -1
meilleurs_parametres["verbosity"] = 0

modele_final = xgb.XGBClassifier(**meilleurs_parametres)
modele_final.fit(X_train, y_train)

y_proba_final = modele_final.predict_proba(X_val)[:, 1]
auc_pr_final = average_precision_score(y_val, y_proba_final)
auc_roc_final = roc_auc_score(y_val, y_proba_final)

print(f"AUC-PR (validation, modèle final) : {auc_pr_final:.4f}")
print(f"AUC-ROC (validation, modèle final) : {auc_roc_final:.4f}")

## 6. SAUVEGARDE DU MODÈLE ET DES RÉSULTATS

In [ ]:
CHEMIN_MODELE = f"{DOSSIER}/modele_xgboost_optimise.joblib"
joblib.dump(modele_final, CHEMIN_MODELE)
print(f"\nModèle sauvegardé : {CHEMIN_MODELE}")

CHEMIN_HISTORIQUE = f"{DOSSIER}/historique_optimisation_optuna.csv"
historique_utile.to_csv(CHEMIN_HISTORIQUE, index=False)
print(f"Historique des essais sauvegardé : {CHEMIN_HISTORIQUE}")

print("""
NOTE MÉTHODOLOGIQUE (à reprendre dans le rapport) :
Une recherche d'hyperparamètres par optimisation bayésienne (Optuna,
40 essais) a été menée sur le modèle XGBoost avec la stratégie de coût
pondéré (meilleure combinaison identifiée lors de la comparaison
systématique). Le gain obtenu par rapport aux hyperparamètres par défaut
est négligeable et le score final reste très en-deçà du seuil de
performance visé (AUC-PR >= 0.90). Ce résultat corrobore le diagnostic
établi lors de l'analyse exploratoire approfondie : l'absence de lien
statistique entre les variables disponibles et la variable cible ne peut
être compensée ni par le choix du modèle, ni par la stratégie de gestion
du déséquilibre, ni par l'optimisation des hyperparamètres.
""")